# We are to pick 10 terms for each of the three pillars E/S/G to average across idiosyncratic public interest

# We are to pick terms with (1) stable meaning 2015–2024, (2) high-enough search volume, (3) clear maps to one pillar

# We institute stoplist.yml to avoid terms which act as connecting language in reports so they aren't evaluated as candidate-worthy despite making up a large section of the reports

e.g.
# stopwords.yml
# Connector / function words + generic 10-K boilerplate words we never want as candidate terms

In [ ]:
# What this script does:
# Read 10-K parquet -> tag docs by pillar (lexicon hits) -> build 1-2 gram terms
# Compute manual TF + manual DF/IDF -> weighted_appearances = mean(tf * idf)
# Write versioned MySQL tables per pillar + save PNG bar charts (word vs weighted appearances)

from __future__ import annotations

from pathlib import Path
import os
import math
from urllib.parse import quote_plus
from datetime import datetime, timezone

import polars as pl
import yaml
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt


# --------------------
# Config
# --------------------
PARQUET_FILE = "spy_10k_2015_present.parquet"
LEXICON_FILE = "ESG_Lexicon.yml"
STOPWORDS_FILE = "stopwords.yml"

START_YEAR = 2015
END_YEAR = 2024

# What this does: bounds compute cost (increase if you have RAM/CPU)
MAX_DOCS_PER_PILLAR = 5000

# Manual TF-IDF parameters
NGRAM_MAX = 2                 # 1-2 grams
MIN_DF = 10                   # term must appear in >=10 docs
MAX_DF_SHARE = 0.70           # drop terms that appear in >70% of docs (boilerplate)
TOP_K = 75                    # shortlist size per pillar

# Output
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TABLE_PREFIX_E = "candidate_terms_E_"
TABLE_PREFIX_S = "candidate_terms_S_"
TABLE_PREFIX_G = "candidate_terms_G_"
RUNS_TABLE = "candidate_terms_runs"


# --------------------
# Helpers
# --------------------
def rm_if_exists(p: Path) -> None:
    if p.exists():
        p.unlink()

def get_engine():
    load_dotenv()
    user = os.getenv("MYSQL_USER", "root")
    pwd = os.getenv("MYSQL_PASSWORD", "")
    host = os.getenv("MYSQL_HOST", "localhost")
    port = os.getenv("MYSQL_PORT", "3306")
    db = os.getenv("MYSQL_DB", "fyp")
    return create_engine(f"mysql+pymysql://{user}:{quote_plus(pwd)}@{host}:{port}/{db}"), db

def pillar_hits_expr(terms: list[str], text_col: str = "text") -> pl.Expr:
    return pl.sum_horizontal([pl.col(text_col).str.count_matches(t) for t in terms])

def next_version(existing_tables: list[str], prefix: str) -> int:
    nums: list[int] = []
    for t in existing_tables:
        if t.startswith(prefix):
            try:
                nums.append(int(t.replace(prefix, "")))
            except ValueError:
                pass
    return (max(nums) + 1) if nums else 1

def add_meta(tbl: pl.DataFrame, pillar: str, run_id: int) -> pl.DataFrame:
    return (
        tbl.with_row_index(name="rank", offset=1)
           .with_columns(
                pl.lit(pillar).alias("pillar"),
                pl.lit(run_id).cast(pl.Int32).alias("run_id"),
                pl.lit(datetime.now(timezone.utc).isoformat()).alias("run_utc"),
                pl.col("rank").cast(pl.Int32),
           )
    )

def plot_top_terms(tbl: pl.DataFrame, pillar_label: str, version: int, top_n: int = 25) -> None:
    top = (
        tbl.select(["term", "weighted_appearances"])
           .sort("weighted_appearances", descending=True)
           .head(top_n)
           .sort("weighted_appearances", descending=False)
           .to_pandas()
    )

    fig_path = OUT_DIR / f"03_terms_{pillar_label}_top{top_n}_{version:02d}.png"
    rm_if_exists(fig_path)

    plt.figure()
    plt.barh(top["term"], top["weighted_appearances"])
    plt.xlabel("Weighted number of appearances")
    plt.ylabel("Word")
    plt.title(f"{pillar_label} pillar — weighted term intensity (top {top_n})")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.close()

    print(f"Saved figure → {fig_path}")


# --------------------
# Load stopwords (YAML)
# --------------------
# What this does: load stopwords safely even if YAML coerces on/off/yes/no into booleans
with open(STOPWORDS_FILE, "r", encoding="utf-8") as f:
    stop_cfg = yaml.safe_load(f) or {}

raw = stop_cfg.get("stopwords") or []

STOPWORDS = set()
for w in raw:
    if isinstance(w, bool):
        w = "on" if w else "off"
    w = str(w).strip().lower()
    if w:
        STOPWORDS.add(w)

assert STOPWORDS, f"Stopwords list is empty or missing in {STOPWORDS_FILE}"


STOPWORDS = set(stop_cfg.get("stopwords") or [])
assert STOPWORDS, f"Stopwords list is empty or missing in {STOPWORDS_FILE}"


# --------------------
# Load lexicon (YAML)
# --------------------
with open(LEXICON_FILE, "r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

E_terms = list(lex.get("environmental") or [])
S_terms = list(lex.get("social") or [])
G_terms = list(lex.get("governance") or [])
assert E_terms and S_terms and G_terms, "Lexicon pillars are empty or missing keys"


# --------------------
# Read parquet (minimal) + year filter
# --------------------
df = pl.read_parquet(PARQUET_FILE, columns=["filing_date", "text"]).with_columns(
    pl.col("filing_date").cast(pl.Date),
    pl.col("filing_date").dt.year().alias("year"),
    pl.col("text").cast(pl.Utf8).fill_null(""),
).filter(
    (pl.col("year") >= START_YEAR) & (pl.col("year") <= END_YEAR)
)

# Tag pillar relevance by lexicon hits
df = df.with_columns(
    pillar_hits_expr(E_terms).alias("E_hits"),
    pillar_hits_expr(S_terms).alias("S_hits"),
    pillar_hits_expr(G_terms).alias("G_hits"),
)

# What this does: assign a stable doc_id and sample per pillar for tractability
df = df.with_row_index("doc_id")

seed = 42

def sample_docs(pillar_col: str) -> pl.DataFrame:
    sub = df.filter(pl.col(pillar_col) > 0).select(["doc_id", "text"])
    n = min(MAX_DOCS_PER_PILLAR, sub.height)
    return sub.sample(n=n, seed=seed, with_replacement=False)

E_docs_df = sample_docs("E_hits")
S_docs_df = sample_docs("S_hits")
G_docs_df = sample_docs("G_hits")

assert E_docs_df.height > 0 and S_docs_df.height > 0 and G_docs_df.height > 0, "One or more pillars has 0 docs"


# --------------------
# Manual TF-IDF (Polars)
# --------------------
def compute_manual_tfidf_terms(docs_df: pl.DataFrame, stopwords: set[str]) -> pl.DataFrame:
    # docs_df: columns [doc_id, text]

    # Tokenise -> explode tokens
    tokens = (
        docs_df
        .with_columns(
            # Extract alphabetic words, lowercase
            pl.col("text").str.to_lowercase().str.extract_all(r"[a-z]+").alias("tok")
        )
        .select(["doc_id", "tok"])
        .explode("tok")
        .rename({"tok": "token"})
        .filter(pl.col("token").is_not_null() & (pl.col("token").str.len_chars() >= 2))
        .filter(~pl.col("token").is_in(list(stopwords)))
    )

    # Unigrams: term counts per doc
    uni_counts = (
        tokens
        .group_by(["doc_id", "token"])
        .agg(pl.len().alias("n"))
        .rename({"token": "term"})
    )

    # Bigrams: create within-doc consecutive pairs
    bigrams = (
        tokens
        .sort(["doc_id"])  # stable
        .with_columns(
            pl.col("token").shift(-1).over("doc_id").alias("token_next")
        )
        .filter(pl.col("token_next").is_not_null())
        .with_columns(
            (pl.col("token") + pl.lit(" ") + pl.col("token_next")).alias("term")
        )
        .select(["doc_id", "term"])
        .group_by(["doc_id", "term"])
        .agg(pl.len().alias("n"))
    )

    # Combine uni + bi (if NGRAM_MAX == 2)
    term_counts = pl.concat([uni_counts, bigrams], how="vertical")

    # TF normalisation: n / total_terms_in_doc (per doc)
    term_counts = term_counts.join(
        term_counts.group_by("doc_id").agg(pl.sum("n").alias("doc_terms")),
        on="doc_id",
        how="left",
    ).with_columns(
        (pl.col("n") / pl.col("doc_terms")).alias("tf")
    )

    # DF: number of docs containing term
    df_tbl = term_counts.group_by("term").agg(pl.n_unique("doc_id").alias("df"))

    N = docs_df.select(pl.n_unique("doc_id").alias("N")).item()
    max_df = int(math.floor(MAX_DF_SHARE * N))

    # Filter terms by DF thresholds
    df_tbl = df_tbl.filter((pl.col("df") >= MIN_DF) & (pl.col("df") <= max_df))

    # IDF: log((N + 1) / (df + 1)) + 1  (smooth + positive)
    df_tbl = df_tbl.with_columns(
        (pl.lit(N + 1) / (pl.col("df") + 1)).log().alias("idf_log_ratio")
    ).with_columns(
        (pl.col("idf_log_ratio") + 1.0).alias("idf")
    ).drop("idf_log_ratio")

    # Join idf back, compute tf-idf per (doc, term)
    tfidf = (
        term_counts
        .join(df_tbl.select(["term", "df", "idf"]), on="term", how="inner")
        .with_columns(
            (pl.col("tf") * pl.col("idf")).alias("tfidf")
        )
    )

    # Weighted appearance score: mean(tfidf) across docs (pillar-internal weighting)
    scored = (
        tfidf
        .group_by("term")
        .agg(
            pl.mean("tfidf").alias("weighted_appearances"),
            pl.first("df").alias("df"),
            pl.first("idf").alias("idf"),
        )
        .sort("weighted_appearances", descending=True)
        .head(TOP_K)
    )

    return scored

E_top = compute_manual_tfidf_terms(E_docs_df, STOPWORDS)
S_top = compute_manual_tfidf_terms(S_docs_df, STOPWORDS)
G_top = compute_manual_tfidf_terms(G_docs_df, STOPWORDS)


# --------------------
# Write versioned MySQL tables + run log
# --------------------
engine, db = get_engine()

with engine.connect() as conn:
    existing_tables = [r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()]

vE = next_version(existing_tables, TABLE_PREFIX_E)
vS = next_version(existing_tables, TABLE_PREFIX_S)
vG = next_version(existing_tables, TABLE_PREFIX_G)
run_id = max(vE, vS, vG)

table_E = f"{TABLE_PREFIX_E}{vE:02d}"
table_S = f"{TABLE_PREFIX_S}{vS:02d}"
table_G = f"{TABLE_PREFIX_G}{vG:02d}"

E_out = add_meta(E_top, "E", run_id)
S_out = add_meta(S_top, "S", run_id)
G_out = add_meta(G_top, "G", run_id)

with engine.begin() as conn:
    conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS {RUNS_TABLE} (
            run_id INT NOT NULL,
            run_utc VARCHAR(32) NOT NULL,
            start_year INT NOT NULL,
            end_year INT NOT NULL,
            max_docs_per_pillar INT NOT NULL,
            top_k INT NOT NULL,
            ngram_max INT NOT NULL,
            min_df INT NOT NULL,
            max_df_share DOUBLE NOT NULL,
            stopwords_file VARCHAR(255) NOT NULL,
            lexicon_file VARCHAR(255) NOT NULL,
            table_E VARCHAR(64) NOT NULL,
            table_S VARCHAR(64) NOT NULL,
            table_G VARCHAR(64) NOT NULL,
            PRIMARY KEY (run_id, run_utc)
        )
    """))

E_out.write_database(table_name=table_E, connection=engine, if_table_exists="fail")
S_out.write_database(table_name=table_S, connection=engine, if_table_exists="fail")
G_out.write_database(table_name=table_G, connection=engine, if_table_exists="fail")

with engine.begin() as conn:
    conn.execute(
        text(f"""
            INSERT INTO {RUNS_TABLE} (
                run_id, run_utc, start_year, end_year, max_docs_per_pillar, top_k,
                ngram_max, min_df, max_df_share, stopwords_file, lexicon_file,
                table_E, table_S, table_G
            )
            VALUES (
                :run_id, :run_utc, :start_year, :end_year, :max_docs, :top_k,
                :ngram_max, :min_df, :max_df_share, :stopwords_file, :lexicon_file,
                :table_E, :table_S, :table_G
            )
        """),
        dict(
            run_id=run_id,
            run_utc=datetime.now(timezone.utc).isoformat(),
            start_year=START_YEAR,
            end_year=END_YEAR,
            max_docs=MAX_DOCS_PER_PILLAR,
            top_k=TOP_K,
            ngram_max=NGRAM_MAX,
            min_df=MIN_DF,
            max_df_share=float(MAX_DF_SHARE),
            stopwords_file=STOPWORDS_FILE,
            lexicon_file=LEXICON_FILE,
            table_E=table_E,
            table_S=table_S,
            table_G=table_G,
        ),
    )

print(f"Wrote → {db}.{table_E}")
print(f"Wrote → {db}.{table_S}")
print(f"Wrote → {db}.{table_G}")
print(f"Logged → {db}.{RUNS_TABLE} (run_id={run_id})")


# --------------------
# PNG outputs (word vs weighted appearances)
# --------------------
plot_top_terms(E_top, "Environmental", vE, top_n=25)
plot_top_terms(S_top, "Social", vS, top_n=25)
plot_top_terms(G_top, "Governance", vG, top_n=25)

TypeError: unexpected value while building Series of type Boolean; found value of type String: "for"

Hint: Try setting `strict=False` to allow passing data with mixed types.